# QuBridge - Logical Teleportation Walkthrough

End-to-end walkthrough of the [[2,1,1]] repetition-encoded six-qubit logical teleportation circuit. We construct the circuit, select a layout on IBM Torino via VF2, build the noise model, run a density-matrix simulation, and compute the conditional logical fidelity.

Reproduces Paper Table V (noise=1.0, |+⟩, logical): Phys F = 0.9849 / Log F = 0.9769 / Accept = 0.9263.

The circuit is `qubridge_logical.create_deferred_logical_teleportation_circuit` (vendored from `services.mode3_computation`). It follows the original Ltelepo Method 2 protocol (CX_L + H_L decode-encode), recast in deferred-measurement form so it can be passed directly to a density-matrix simulator.

In [ ]:
import sys, os, math
# Walkthrough imports use the vendored `qubridge_logical/` package next to
# this notebook's parent directory. Add the artifact root to sys.path.
_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import numpy as np
from qiskit import transpile
from qiskit.transpiler import CouplingMap
from qiskit_aer import AerSimulator

from qubridge_logical import (
    create_deferred_logical_teleportation_circuit,
    compute_dm_fidelity_logical,
    compute_dm_fidelity_physical,
    select_qubits_qubridge_logical,
    create_reduced_noise_model,
    DEFAULT_GATE_PULSE_MAP,
    load_static_backend_data,
)

## 1. Input state and circuit construction

We prepare the stress-test state |+⟩ (θ=π/2, φ=0). Roles of the six logical qubits:

| index | role |
|---|---|
| 0, 1 | Alice (encoded |ψ_L⟩) |
| 2, 3 | Mediator (one half of the encoded Bell pair) |
| 4, 5 | Bob (entangled with Mediator) |

In [ ]:
theta = math.pi / 2  # |+⟩
phi = 0.0
qc = create_deferred_logical_teleportation_circuit(theta, phi)
print(f'Qubits: {qc.num_qubits}')
_pre_ops = qc.count_ops()
print(f'Pre-transpile 2Q gates: {_pre_ops.get("cx", 0) + _pre_ops.get("cz", 0)}')
qc.draw('mpl', fold=-1)

## 2. VF2 layout selection on IBM Torino

VF2 enumerates all six-qubit subgraphs of the heavy-hex coupling map that satisfy the circuit's connectivity template, then picks the one with the best noise score. The template is a tree with 5 edges: Alice (0,1), Alice→Med (1,2), Med (2,3), Med→Bob (3,4), and Med→Bob (3,5).

In [ ]:
backend_name = 'ibm_torino'
# Match E3DConfig (threshold=1.0, seed=42) for reproducibility
selected = select_qubits_qubridge_logical(backend_name, threshold=1.0, seed=42)
labels = ['Alice_0', 'Alice_1', 'Med_0', 'Med_1', 'Bob_0', 'Bob_1']
for lbl, q in zip(labels, selected):
    print(f'  {lbl:8s} (logical) -> physical {q}')

## 3. Transpile and inspect routing SWAPs

VF2 only guarantees the 5-edge tree template. The circuit needs more 2Q connections than that, so the transpiler inserts SWAPs to route the missing edges.

In [ ]:
data = load_static_backend_data(backend_name)
cm = CouplingMap(couplinglist=data['coupling_map'])
basis = ['sx', 'x', 'rz', 'cz', 'id', 'measure']
qct = transpile(qc, basis_gates=basis, coupling_map=cm,
                initial_layout=selected, optimization_level=3,
                seed_transpiler=42)
ops = qct.count_ops()
print(f'Post-transpile gate counts: {dict(ops)}')
print(f'Post-transpile 2Q (cz) gates: {ops.get("cz", 0)}')

In [ ]:
qct.draw('mpl', fold=120)

## 4. Build the noise model (Lindblad synthesis)

A reduced model that injects calibrated noise only on the selected physical qubits. For each gate we compose `thermal_relaxation_error` (T1, T2) with `PauliLindbladError` (RB-anchored Pauli rates).

In [ ]:
noise_scale = 1.0  # calibration anchor used in Paper Table V
noise_model = create_reduced_noise_model(backend_name, selected, noise_scale=noise_scale)
print(f'Noise model gates: {noise_model.noise_instructions}')
print(f'Noise scale: {noise_scale} (1.0 = device calibration)')

## 5. Density-matrix simulation and fidelity

`compute_dm_fidelity_logical` internally:

1. Transpiles with a reduced coupling map so SWAPs stay within the selected physical qubits
2. Runs `AerSimulator(method='density_matrix')` to obtain the full density matrix
3. Post-selects on the syndrome subspace ({00, 11} ⊗ {00, 11})
4. Takes the partial trace over Bob and compares against the ideal logical Bob state

In [ ]:
result = compute_dm_fidelity_logical(
    theta, phi,
    noise_model=noise_model,
    backend_name=backend_name,
    selected_qubits=selected,
    gate_pulse_map=DEFAULT_GATE_PULSE_MAP,
    noise_scale=noise_scale,
)
print(f'Logical F (conditional)     : {result["fidelity"]:.4f}')
print(f'Error detection rate        : {result["error_detection_rate"]:.4f}')
print(f'Acceptance (1 - EDR)        : {1 - result["error_detection_rate"]:.4f}')

## 6. Compare with physical (3-qubit) teleportation

Run physical teleportation at the same noise scale and inspect the logical-vs-physical gap.

In [ ]:
phys_qubits = [66, 67, 74]  # E3DConfig.best_path
phys_noise = create_reduced_noise_model(backend_name, phys_qubits, noise_scale=noise_scale)
phys_result = compute_dm_fidelity_physical(
    theta, phi,
    noise_model=phys_noise,
    backend_name=backend_name,
    selected_qubits=phys_qubits,
    gate_pulse_map=DEFAULT_GATE_PULSE_MAP,
)
print(f'Physical F                  : {phys_result["fidelity"]:.4f}')
print(f'Logical F (conditional)     : {result["fidelity"]:.4f}')
print(f'Delta F (Log - Phys)        : {result["fidelity"] - phys_result["fidelity"]:+.4f}')

## 7. Cross-check against Paper Table V

Paper §IV.D Table V, row noise=1.0, |+⟩:

| | Phys F | Log F | Accept |
|---|---|---|---|
| Paper claim | 0.9849 | 0.9769 | 0.9263 |
| This run | (above) | (above) | (above) |

If the numbers match, the implementation and the paper's claim are contiguous. For |+⟩, Log F < Phys F is expected because the [[2,1,1]] code does not detect Z errors, giving ΔF ≈ -0.008.

In [ ]:
paper = {'Phys F': 0.9849, 'Log F': 0.9769, 'Accept': 0.9263}
this_run = {
    'Phys F': phys_result['fidelity'],
    'Log F': result['fidelity'],
    'Accept': 1 - result['error_detection_rate'],
}
print(f'{"":>10} {"Paper":>10} {"This run":>10} {"Diff":>10}')
for k in paper:
    diff = this_run[k] - paper[k]
    print(f'{k:>10} {paper[k]:>10.4f} {this_run[k]:>10.4f} {diff:>+10.4f}')